In [1]:
!pip install transformers

import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
df = pd.read_csv("/kaggle/input/datasets/venkatsaikondra/reddit/r_dataisbeautiful_posts.csv", low_memory=False)

df = df[["title", "num_comments", "created_utc"]]
df = df.dropna()

print(df.shape)

(166218, 3)


In [3]:
df["num_comments"] = pd.to_numeric(df["num_comments"], errors="coerce")
df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")

df = df.dropna()

In [4]:
current_time = time.time()

df["age_hours"] = (current_time - df["created_utc"]) / 3600
df["log_comments"] = np.log1p(df["num_comments"])

In [5]:
threshold = df["num_comments"].quantile(0.80)
df["viral"] = (df["num_comments"] >= threshold).astype(int)

In [6]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [15]:
val_dataset = ViralDataset(val_df)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
class ViralDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        text = row["title"]
        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=64,
            return_tensors="pt"
        )
        
        numeric = torch.tensor([
            row["age_hours"],
            row["log_comments"]
        ], dtype=torch.float32)
        
        label = torch.tensor(row["viral"], dtype=torch.float32)
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "numeric": numeric,
            "label": label
        }

In [16]:
train_dataset = ViralDataset(train_df)
val_dataset   = ViralDataset(val_df)
test_dataset  = ViralDataset(test_df)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_dataset, batch_size=32)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=32)

In [10]:
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
        
        self.numeric_net = nn.Sequential(
            nn.Linear(2, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(768 + 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids, attention_mask, numeric):
        text_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = text_out.last_hidden_state[:, 0, :]
        
        num_out = self.numeric_net(numeric)
        
        combined = torch.cat((cls, num_out), dim=1)
        
        return self.classifier(combined)

In [11]:
model = MultiModalModel().to(device)
pos_weight = torch.tensor([4.0]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=1, gamma=0.7
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
for param in model.bert.parameters():
    param.requires_grad = False

In [13]:
for epoch in range(10):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)
        labels = batch["label"].to(device).unsqueeze(1)
        
        outputs = model(input_ids, attention_mask, numeric)
        
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    scheduler.step()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss}")

Epoch 1, Loss: 3795.1574655771255
Epoch 2, Loss: 3153.4328095018864
Epoch 3, Loss: 2593.1569977253675
Epoch 4, Loss: 2187.2554113790393
Epoch 5, Loss: 1974.9258522987366
Epoch 6, Loss: 1676.1857360862195
Epoch 7, Loss: 1428.6780605344102
Epoch 8, Loss: 1396.6777151888236
Epoch 9, Loss: 1320.6057664798573
Epoch 10, Loss: 1273.523741436191


In [18]:
from sklearn.metrics import f1_score

y_true = []
y_probs = []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)

        outputs = model(input_ids, attention_mask, numeric)
        probs = torch.sigmoid(outputs).cpu().numpy()

        y_probs.extend(probs.flatten())
        y_true.extend(batch["label"].numpy())

# find best threshold
best_f1 = 0
best_thresh = 0.5

for t in np.arange(0.1, 0.9, 0.05):
    preds = (np.array(y_probs) > t).astype(int)
    f1 = f1_score(y_true, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best Threshold:", best_thresh)
print("Validation F1:", best_f1)

Best Threshold: 0.8500000000000002
Validation F1: 0.7582658200131377


In [19]:
from sklearn.metrics import classification_report

y_true = []
y_probs = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric = batch["numeric"].to(device)

        outputs = model(input_ids, attention_mask, numeric)
        probs = torch.sigmoid(outputs).cpu().numpy()

        y_probs.extend(probs.flatten())
        y_true.extend(batch["label"].numpy())

final_preds = (np.array(y_probs) > best_thresh).astype(int)

print(classification_report(y_true, final_preds))

              precision    recall  f1-score   support

         0.0       1.00      0.83      0.91     13302
         1.0       0.59      1.00      0.75      3320

    accuracy                           0.86     16622
   macro avg       0.80      0.92      0.83     16622
weighted avg       0.92      0.86      0.87     16622



In [22]:
def predict_viral(title, comments, age_hours):
    
    encoding = tokenizer(
        title,
        truncation=True,
        padding="max_length",
        max_length=64,
        return_tensors="pt"
    )
    
    numeric = torch.tensor(
        [[age_hours, np.log1p(comments)]],
        dtype=torch.float32
    ).to(device)
    
    with torch.no_grad():
        output = model(
            encoding["input_ids"].to(device),
            encoding["attention_mask"].to(device),
            numeric
        )
    
    prob = torch.sigmoid(output).item()
    
    return prob, int(prob > best_thresh)

In [23]:
prob, label = predict_viral(
    "Amazing data visualization about global warming",
    50,
    2
)

print("Probability:", prob)
print("Predicted:", label)

Probability: 0.9980165958404541
Predicted: 1


In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df[["age_hours", "log_comments"]] = scaler.fit_transform(
    df[["age_hours", "log_comments"]]
)

In [30]:
import torch
import joblib

import os

os.makedirs("/kaggle/working/model", exist_ok=True)

# 1. Save model weights
torch.save(model.state_dict(), "/kaggle/working/model/model.pt")

# 2. Save tokenizer
tokenizer.save_pretrained("/kaggle/working/model/tokenizer")

# 3. Save scaler (if you used StandardScaler)
joblib.dump(scaler, "/kaggle/working/model/scaler.pkl")

# 4. Save threshold
with open("/kaggle/working/model/threshold.txt", "w") as f:
    f.write(str(best_thresh))

print("✅ Model saved successfully")

✅ Model saved successfully


In [31]:
import torch
import joblib
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
import torch
import torch.nn as nn
from transformers import AutoModel
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import numpy as np
import joblib
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
        
        self.numeric_net = nn.Sequential(
            nn.Linear(2, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(768 + 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids, attention_mask, numeric):
        text_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = text_out.last_hidden_state[:, 0, :]
        
        num_out = self.numeric_net(numeric)
        
        combined = torch.cat((cls, num_out), dim=1)
        
        return self.classifier(combined)

In [34]:
model.load_state_dict(
    torch.load("/kaggle/working/model/model.pt", map_location=device)
)

<All keys matched successfully>

In [36]:
import os
print(os.listdir("/kaggle/working/model"))

['model.pt', 'scaler.pkl', 'tokenizer', 'threshold.txt']


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalModel().to(device)

model.load_state_dict(
    torch.load("model/model.pt", map_location=device)
)

model.eval()

print("✅ Model loaded successfully")

✅ Model loaded successfully


In [39]:
import numpy as np

def predict_viral(title, comments, age_hours):
    
    # Text processing
    encoding = tokenizer(
        title,
        truncation=True,
        padding="max_length",
        max_length=64,
        return_tensors="pt"
    )
    
    # Numeric processing
    numeric = np.array([[age_hours, np.log1p(comments)]])
    numeric = scaler.transform(numeric)
    numeric = torch.tensor(numeric, dtype=torch.float32).to(device)
    
    # Prediction
    with torch.no_grad():
        output = model(
            encoding["input_ids"].to(device),
            encoding["attention_mask"].to(device),
            numeric
        )
    
    prob = torch.sigmoid(output).item()
    label = int(prob > best_thresh)
    
    return prob, label

In [49]:
prob, label = predict_viral(
    "Amazing data visualization about global warming",
    comments=13,
    age_hours=2
)

print("Probability:", prob)
print("Predicted:", label)

Probability: 0.8563237190246582
Predicted: 1


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
